In [50]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
df = pd.read_json("review_summary.json")
print(df)

       review_id  user_id  res_id  rating
0              0        1       0     9.4
1              1        2       0     6.8
2              2        3       0     8.4
3              3        4       0    10.0
4              4        5       0     8.6
...          ...      ...     ...     ...
39530      39530     1871    1398     7.8
39531      39531     4887    1398     7.0
39532      39532      161    1398     7.0
39533      39533     6419    1398     7.0
39534      39534    17129    1398     7.6

[39535 rows x 4 columns]


In [41]:
rating_matrix = df.pivot_table(index='user_id', columns='res_id', values='rating')
print(rating_matrix)

res_id   0     1     2     3     4     5     6     7     8     9     ...  \
user_id                                                              ...   
1         9.4   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
2         6.8   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
3         8.4   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4        10.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
5         8.6   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
...       ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
17125     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
17126     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
17127     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
17128     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
17129     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

res_id   13

In [43]:
user_similarity = pd.DataFrame(
    cosine_similarity(rating_matrix.fillna(0)),
    index=rating_matrix.index,
    columns=rating_matrix.index
)
print(user_similarity)

user_id     1         2         3         4         5         6         7      \
user_id                                                                         
1        1.000000  0.129831  0.426480  0.426480  0.237828  0.148918  0.246148   
2        0.129831  1.000000  0.304425  0.304425  0.118267  0.106299  0.102231   
3        0.426480  0.304425  1.000000  1.000000  0.388493  0.349180  0.335817   
4        0.426480  0.304425  1.000000  1.000000  0.388493  0.349180  0.335817   
5        0.237828  0.118267  0.388493  0.388493  1.000000  0.135654  0.130462   
...           ...       ...       ...       ...       ...       ...       ...   
17125    0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
17126    0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
17127    0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
17128    0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
17129    0.000000  0.000000 

In [44]:
def predict_rating(user_id, res_id, k=5):
    # Danh sách người dùng đã đánh giá nhà hàng đó
    users_rated = rating_matrix[rating_matrix[res_id].notna()].index
    
    # Độ tương tự giữa user hiện tại và các user khác
    sims = user_similarity.loc[user_id, users_rated]
    
    # Lấy top K người giống nhất
    sims = sims.sort_values(ascending=False)[:k]
    
    # Điểm rating của họ
    ratings = rating_matrix.loc[sims.index, res_id]
    
    # Tính điểm dự đoán có trọng số
    if sims.sum() == 0:
        return np.nan
    pred = np.dot(sims, ratings) / sims.sum()
    return pred

In [46]:
def recommend_for_user(user_id, n=5):
    # Các quán user này chưa đánh giá
    unrated_items = rating_matrix.columns[rating_matrix.loc[user_id].isna()]
    preds = []
    
    for item in unrated_items:
        est = predict_rating(user_id, item)
        if not np.isnan(est):
            preds.append((item, est))
    
    preds.sort(key=lambda x: x[1], reverse=True)
    return preds[:n]

In [49]:
recommendations = recommend_for_user(1000, n=5)

print("Top gợi ý cho user 10:")
for res_id, score in recommendations:
    print(f" - Nhà hàng {res_id}: dự đoán {score:.2f}")

Top gợi ý cho user 10:
 - Nhà hàng 374: dự đoán 10.00
 - Nhà hàng 729: dự đoán 10.00
 - Nhà hàng 745: dự đoán 9.71
 - Nhà hàng 1081: dự đoán 9.40
 - Nhà hàng 787: dự đoán 9.20
